# Gdańsk Regional Championship: Day 2 team analysis

This notebook explores the 110 teams above the **Day 2 end** divider in the Gdańsk Regional Championship standings. It compares team compositions, groups teams connected by an exact similarity score, and demonstrates a damage matchup.

The Gdańsk dataset is stored at `m-c/gdansk_day_2.json`. Run this notebook with the `showcases` folder as the working directory. The `pyvgc` package must be installed in the active Python environment.

In [ ]:
import json
from pyvgc import (
    group_identical_teamplayers,
    pairwise_score_matrix,
)

with open("m-c/regional-louisville-ky-day-2.json", encoding="utf-8") as data_file:
    standings_data = json.load(data_file)

teams = {
    entry["rank"]: {"player": entry["player"], "team": entry["team"]}
    for entry in standings_data[0:]
}
print(f"Loaded {len(teams)} teams ({sum(len(t['team']) for t in teams.values())} Pokémon sets).")
print("Standings source: https://standings.limitlessvgc.com/0044/standings")

## Inspect the standings teams

Each record includes the player name and six sets containing Pokémon, item, ability, stat alignment (represented as nature by the package), and four moves.

In [ ]:
for rank, entry in teams.items():
    roster = ", ".join(pokemon["name"] for pokemon in entry["team"])
    print(f"{rank:>3}. {entry['player']}: {roster}")

## Compare team similarity

The score starts at 102 for two identical six-Pokémon teams.

More documentation about this function in the future.

A **higher score means more similar teams**. This implementation accepts the raw website team dictionaries, so it can also analyze sets that are outside the package's current M-C legality snapshot.

In [ ]:
score_matrix = pairwise_score_matrix(teams)
column_width = max(4, len(str(max(teams))) + 1)

print("Rows and columns are standings ranks; identical teams score 102.")
print(" " * column_width + "".join(f"{rank:>{column_width}}" for rank in teams))
for rank, scores in score_matrix.items():
    print(f"{rank:>{column_width}}" + "".join(
        f"{scores[other_rank]:>{column_width}}" for other_rank in teams
    ))
print(f"Validated a symmetric {len(teams)} × {len(teams)} score matrix.")

## Find score-linked groups

A pair is included only when its score equals the selected score. Groups are connected components: if A matches B and B matches C at that score.

## Exact-score pairs and player groups

Team comparison, pairwise matrix generation, exact-score pair lookup, and transitive player grouping are provided by the public `pyvgc` API. Grouping follows connected components: a chain of matching pairs joins its members even if not every pair within a group directly matches.

In [ ]:
target_score_pairs = 102

score_groups = group_identical_teamplayers(score_matrix)
print("\nGroups of players with identical teams (score 102):")
for group in score_groups:
    members = ", ".join(
        f"{rank} ({teams[rank]['player']})" for rank in group
    )
    print(members)

## Pokémon usage and set statistics

Pokémon usage is the percentage of these teams that brought each Pokémon. Nature, item, move, and exact-moveset percentages are calculated among that Pokémon's appearances. Teammate percentages are conditional: among teams that brought the selected Pokémon, the percentage that also brought each teammate. Forms are counted separately as named in the source data.

In [ ]:
from collections import Counter, defaultdict

team_count = len(standings_data)
pokemon_team_counts = Counter()
sets_by_pokemon = defaultdict(list)
teammate_team_counts = defaultdict(Counter)

for entry in standings_data:
    members = entry["team"]
    pokemon_names = [member["name"] for member in members]
    if len(pokemon_names) != 6 or len(set(pokemon_names)) != 6:
        raise ValueError(
            f"Rank {entry['rank']} must have six distinct Pokémon"
        )
    pokemon_team_counts.update(pokemon_names)
    for member in members:
        sets_by_pokemon[member["name"]].append(member)
        teammate_team_counts[member["name"]].update(
            teammate for teammate in pokemon_names
            if teammate != member["name"]
        )

if sum(pokemon_team_counts.values()) != team_count * 6:
    raise ValueError("Expected exactly six Pokémon per team")


def print_table(title, headers, rows):
    formatted_rows = []
    for row in rows:
        formatted_rows.append(
            [
                f"{float(value):.1f}%" if header.endswith("%") else str(value)
                for header, value in zip(headers, row)
            ]
        )
    widths = [
        max(len(header), *(len(row[index]) for row in formatted_rows))
        for index, header in enumerate(headers)
    ]
    print(f"\n{title}")
    print(" | ".join(header.ljust(widths[index]) for index, header in enumerate(headers)))
    print("-+-".join("-" * width for width in widths))
    for row in formatted_rows:
        print(" | ".join(value.ljust(widths[index]) for index, value in enumerate(row)))

pokemon_order = sorted(
    pokemon_team_counts,
    key=lambda name: (-pokemon_team_counts[name], name.casefold()),
)


### Pokémon usage

In [ ]:
usage_rows = [
    (
        pokemon,
        pokemon_team_counts[pokemon],
        pokemon_team_counts[pokemon] / team_count * 100,
    )
    for pokemon in pokemon_order
]
print_table(
    "Pokémon usage across Day 2 teams",
    ["Pokémon", "Teams", "Team usage %"],
    usage_rows,
)


### Nature, item, and movepool usage

Move usage counts each move once per set and reports the percentage of that Pokémon's sets containing it. Exact movesets are also listed as a four-move combination; their percentages sum to 100% for each Pokémon.

In [ ]:
def distribution_rows(counter_by_pokemon):
    rows = []
    for pokemon in pokemon_order:
        denominator = pokemon_team_counts[pokemon]
        for option, count in sorted(
            counter_by_pokemon[pokemon].items(),
            key=lambda item: (-item[1], str(item[0]).casefold()),
        ):
            rows.append(
                (pokemon, option, count, count / denominator * 100)
            )
    return rows


nature_counts = defaultdict(Counter)
item_counts = defaultdict(Counter)
move_counts = defaultdict(Counter)
moveset_counts = defaultdict(Counter)
for pokemon, sets in sets_by_pokemon.items():
    for pokemon_set in sets:
        nature_counts[pokemon][pokemon_set.get("nature") or "Unknown"] += 1
        item_counts[pokemon][pokemon_set.get("item") or "Unknown"] += 1
        move_counts[pokemon].update(set(pokemon_set["moves"]))
        moveset = tuple(sorted(pokemon_set["moves"], key=str.casefold))
        moveset_counts[pokemon][moveset] += 1


def show_distribution(title, label, counter_by_pokemon, format_option=str):
    rows = [
        (pokemon, format_option(option), count, percent)
        for pokemon, option, count, percent in distribution_rows(
            counter_by_pokemon
        )
    ]
    print_table(
        title,
        ["Pokémon", label, "Sets", "Set usage %"],
        rows,
    )


show_distribution("Nature usage by Pokémon", "Nature", nature_counts)
show_distribution("Item usage by Pokémon", "Item", item_counts)
show_distribution("Individual move usage by Pokémon", "Move", move_counts)
show_distribution(
    "Exact four-move sets by Pokémon",
    "Moveset",
    moveset_counts,
    lambda moves: ", ".join(moves),
)


### Teammate usage

In [ ]:
teammate_rows = []
for pokemon in pokemon_order:
    denominator = pokemon_team_counts[pokemon]
    for teammate, count in sorted(
        teammate_team_counts[pokemon].items(),
        key=lambda item: (-item[1], item[0].casefold()),
    ):
        teammate_rows.append(
            (
                pokemon,
                teammate,
                count,
                count / denominator * 100,
            )
        )
print_table(
    "Teammates by Pokémon",
    ["Pokémon", "Teammate", "Teams together", "Teammate usage %"],
    teammate_rows,
)
